# Football Analysis — API trên Colab GPU

> **Khuyến nghị:** Dùng [`colab_tracking.ipynb`](colab_tracking.ipynb) — chỉ tracking trên Colab + **tự chạy khi upload web** (ổn định hơn, ít OOM).

Chạy FastAPI backend trên Colab (GPU) và dùng web React trên máy local.

## Chuẩn bị
1. **Runtime → Change runtime type → T4 GPU**
2. Lấy [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) (miễn phí)
3. Upload `best.pt` lên Google Drive hoặc sẵn trong repo

## Sau khi chạy xong cell cuối
Copy URL ngrok → tạo file `frontend/.env.local` trên máy local:
```
VITE_API_BASE=https://xxxx.ngrok-free.app/api
```
Rồi chạy `npm run dev` trong thư mục `frontend`. Giữ tab Colab mở trong lúc demo.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/PhuongAnh777/football_analysis.git"
BRANCH = "main"  # đổi thành "Collab" nếu cần

if os.path.isdir("football_analysis"):
    %cd football_analysis
    !git pull
else:
    !git clone -b {BRANCH} {REPO_URL}
    %cd football_analysis

In [ ]:
# Colab đã có torch+CUDA — không cài lại torch
!pip install -q fastapi "uvicorn[standard]" python-multipart aiofiles requests python-dotenv \
    ultralytics opencv-python numpy pandas scikit-learn scipy matplotlib pyngrok

In [ ]:
import os

MODEL_PATH = "models/best.pt"
os.makedirs("models", exist_ok=True)

if os.path.exists(MODEL_PATH):
    print("Model OK:", MODEL_PATH)
else:
    USE_DRIVE = True  # False → dùng upload trực tiếp bên dưới
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_MODEL = "/content/drive/MyDrive/football_analysis/best.pt"
        !cp "{DRIVE_MODEL}" "{MODEL_PATH}"
        print("Copied model from Drive")
    else:
        from google.colab import files
        uploaded = files.upload()
        !mv best.pt models/best.pt
        print("Uploaded model")

In [ ]:
import os, sys, socket, time, threading, traceback, subprocess
import urllib.request, json
from pyngrok import ngrok
import uvicorn

# ← điền token từ https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""

if not NGROK_AUTHTOKEN:
    raise ValueError("Hãy điền NGROK_AUTHTOKEN trước khi chạy cell này.")

PORT = 8000
os.environ["CORS_ORIGINS"] = "http://localhost:3000,http://127.0.0.1:3000"

# ── 1. Kiểm tra import trước khi khởi động ───────────────────────────────
print("Kiểm tra import api.main_api...", end="", flush=True)
try:
    import importlib, api.main_api as _m
    importlib.reload(_m)
    print(" OK")
except Exception as _e:
    raise RuntimeError(
        f"Import api.main_api thất bại:\n{traceback.format_exc()}\n\n"
        "Kiểm tra: pip install các package còn thiếu rồi chạy lại."
    )

# ── 2. Kill port cũ + đóng tunnel cũ ────────────────────────────────────
try:
    if subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True, timeout=5).returncode == 0:
        print(f"Đã kill process cũ trên port {PORT}"); time.sleep(1)
except Exception:
    pass
try:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url); print(f"Đã đóng tunnel cũ: {t.public_url}")
except Exception:
    pass

# ── 3. Khởi động uvicorn ─────────────────────────────────────────────────
_server_error = []
def _run_server():
    try:
        uvicorn.run("api.main_api:app", host="0.0.0.0", port=PORT, log_level="info")
    except Exception as e:
        _server_error.append(traceback.format_exc())
        print(f"\n[server-thread] CRASH: {e}", flush=True)

threading.Thread(target=_run_server, daemon=True).start()

print(f"Đang chờ uvicorn bind port {PORT}...", end="", flush=True)
for _i in range(25):
    time.sleep(1)
    if _server_error:
        raise RuntimeError(f"Server crash:\n{_server_error[0]}")
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=0.5):
            print(f" port OK ({_i+1}s)", flush=True); break
    except OSError:
        print(".", end="", flush=True)
else:
    raise RuntimeError("uvicorn không bind port sau 25s → Runtime → Restart runtime")

# ── 4. Kiểm tra health endpoint cục bộ ──────────────────────────────────
print("Kiểm tra /api/health cục bộ...", end="", flush=True)
for _j in range(10):
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/health")
        with urllib.request.urlopen(req, timeout=3) as r:
            _hdata = json.loads(r.read())
            print(f" OK → {_hdata}"); break
    except Exception as _e:
        print(".", end="", flush=True); time.sleep(1)
else:
    raise RuntimeError(
        "uvicorn đang chạy nhưng /api/health không phản hồi sau 10s.\n"
        f"Lỗi cuối: {_e}\nXem log uvicorn bên trên để biết chi tiết."
    )

# ── 5. Tạo ngrok tunnel ──────────────────────────────────────────────────
public_tunnel = ngrok.connect(PORT, bind_tls=True)
public_url = public_tunnel.public_url.rstrip("/")

print("=" * 60)
print("API công khai:  ", public_url)
print("Health check:   ", f"{public_url}/api/health")
print()
print("Trên máy local, tạo file frontend/.env.local:")
print(f"  VITE_API_BASE={public_url}/api")
print()
print("Sau đó: cd frontend && npm run dev")
print("Giữ tab Colab mở trong lúc phân tích.")
print("=" * 60)

In [ ]:
# Kiểm tra API (chạy sau cell server)
import time, json, urllib.request

health_url = f"{public_url}/api/health"
headers = {"ngrok-skip-browser-warning": "true"}

print("Đang chờ server khởi động...", end="")
for attempt in range(15):
    time.sleep(2)
    try:
        req = urllib.request.Request(health_url, headers=headers)
        with urllib.request.urlopen(req, timeout=10) as resp:
            body = resp.read().decode()
            if body.strip():
                data = json.loads(body)
                print(f"\nhealth: {data}")
                break
        print(".", end="", flush=True)
    except Exception:
        print(".", end="", flush=True)
else:
    raise RuntimeError(
        "Server không phản hồi sau 30s — kiểm tra cell server có lỗi không."
    )

print("✓ API sẵn sàng")